In [1]:
# ===== 모델 다운로드: Tri-7B =====
from huggingface_hub import snapshot_download
from pathlib import Path

# 모델을 저장할 local 절대 경로 설정 (~/AI_programming/petmint-rag-chatbot/models/base/Tri-7B)
MODEL_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot" / "models" / "base" / "Tri-7B"

# Hugging Face 허브에서 'trillionlabs/Tri-7B' 모델 전체(가중치, 토크나이저, 설정 등)를 다운로드
# local_dir: 지정한 경로에 디렉터리 구조를 유지하며 파일 다운로드
path = snapshot_download(
    repo_id="trillionlabs/Tri-7B",
    local_dir=str(MODEL_DIR),
#    local_dir_use_symlinks=False,  # 심볼릭 링크 대신 직접 파일 저장을 하면 캐시 디렉터리에 중복으로 파일 쌓이는 것 방지
)
print("다운로드 완료:", path)

/home/kopo17/llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 22 files: 100%|██████████| 22/22 [02:33<00:00,  6.98s/it]

다운로드 완료: /home/kopo17/AI_programming/petmint-rag-chatbot/models/base/Tri-7B


In [3]:
# ===== 모델 다운로드 확인: Tri-7B =====
# 다운로드된 모델 디렉터리의 전체 용량을 확인 (Human-readable format: GB/MB 단위 표시)
!du -sh ~/AI_programming/petmint-rag-chatbot/models/base/Tri-7B/
# 모델 디렉터리 내부의 파일 리스트 확인 (가중치 파일, 토크나이저, config 파일 등 존재 여부 점검)
!ls ~/AI_programming/petmint-rag-chatbot/models/base/Tri-7B/

29G	/home/kopo17/AI_programming/petmint-rag-chatbot/models/base/Tri-7B/
LICENSE				  model-00004-of-00007.safetensors
README.md			  model-00005-of-00007.safetensors
assets				  model-00006-of-00007.safetensors
chat_template.jinja		  model-00007-of-00007.safetensors
config.json			  model.safetensors.index.json
generation_config.json		  special_tokens_map.json
model-00001-of-00007.safetensors  tokenizer.json
model-00002-of-00007.safetensors  tokenizer_config.json
model-00003-of-00007.safetensors


In [5]:
!find / -maxdepth 5 -iname "llama.cpp" -type d 2>/dev/null

/home/kopo17/AI_programming/notebooks/llama.cpp


In [6]:
# ===== 경로 설정: llama.cpp, 모델, 출력 =====
import sys
from pathlib import Path

# llama.cpp 저장소 경로 설정 (~/AI_programming/notebooks/llama.cpp)
LLAMA_CPP_DIR = Path.home() / "AI_programming" / "notebooks" / "llama.cpp"
# 원본 HuggingFace 모델(Tri-7B) 경로 설정
MODEL_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot" / "models" / "base" / "Tri-7B"
# GGUF 변환 결과물을 저장할 디렉터리 경로 설정
GGUF_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot" / "models" / "gguf"
# GGUF 디렉터리가 없으면 생성 (parents=True: 상위 폴더까지 일괄 생성, exist_ok=True: 이미 존재해도 에러 안 남)
GGUF_DIR.mkdir(parents=True, exist_ok=True)
# 변환될 F16 GGUF 파일의 최종 출력 경로 지정
f16_path = GGUF_DIR / "Tri-7B-f16.gguf"
# 필수 경로 존재 여부 출력 (True가 나와야 다음 변환 단계 진행 가능)
print("llama.cpp 존재:", LLAMA_CPP_DIR.exists())
print("모델 폴더 존재:", MODEL_DIR.exists())

llama.cpp 존재: True
모델 폴더 존재: True


In [7]:
# ===== GGUF 변환: safetensors → f16 GGUF =====
# 원본이 29GB라 시간이 좀 걸림. custom code 실행 여부를 묻는 창이 뜨면 자동으로 N 처리됨.
!{sys.executable} {LLAMA_CPP_DIR}/convert_hf_to_gguf.py {MODEL_DIR} --outtype f16 --outfile {f16_path}

INFO:hf-to-gguf:Loading model: Tri-7B
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00007.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float32 --> F16, shape = {4096, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float32 --> F32, shape = {4096}
INFO:hf-t

In [8]:
# ===== 양자화: f16 GGUF → Q4_K_M =====
# EXAONE 때는 exaone.attention.layer_norm_rms_epsilon 키 문제로 --override-kv가 필요했음.
# Tri-7B는 llama 구조라 이 문제가 없을 가능성이 큼. 먼저 override 없이 시도.
q4_path = GGUF_DIR / "Tri-7B-Q4_K_M.gguf"

!{LLAMA_CPP_DIR}/build/bin/llama-quantize {f16_path} {q4_path} Q4_K_M

version: 0.4.1-dev (build 11028, commit 972d2313b)
built with GNU 15.2.0 for Linux x86_64
llama_quantize: quantizing '/home/kopo17/AI_programming/petmint-rag-chatbot/models/gguf/Tri-7B-f16.gguf' to '/home/kopo17/AI_programming/petmint-rag-chatbot/models/gguf/Tri-7B-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 291 tensors from /home/kopo17/AI_programming/petmint-rag-chatbot/models/gguf/Tri-7B-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Tri 7B
llama_model_loader: - kv   3:                           general.basename str              = Tri
llama_model_loader: - kv   4:        

In [9]:
# ===== Ollama 등록: Modelfile 작성 후 컨테이너에 복사, create =====
# Tri-7B는 chat_template.jinja를 갖고 있어 기본 대화 형식을 지원함.
# 파인튜닝 전 원본 상태이므로 SYSTEM은 간단하게, TEMPLATE은 Ollama 기본값에 맡김.

modelfile_path = GGUF_DIR / "Modelfile.tri7b"

with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(f'''FROM {q4_path}
PARAMETER num_ctx 8192
PARAMETER temperature 0.3
''')

print(open(modelfile_path).read())

FROM /home/kopo17/AI_programming/petmint-rag-chatbot/models/gguf/Tri-7B-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.3



In [10]:
# ===== 컨테이너 안으로 복사 후 Ollama에 등록 =====
!docker cp {q4_path} ollama:/tmp/Tri-7B-Q4_K_M.gguf
!docker cp {modelfile_path} ollama:/tmp/Modelfile.tri7b
!docker exec ollama ollama create tri-7b -f /tmp/Modelfile.tri7b
!docker exec ollama ollama list | grep tri

Successfully copied 4.63GB (transferred 4.63GB) to ollama:/tmp/Tri-7B-Q4_K_M.gguf
Successfully copied 133B (transferred 2.05kB) to ollama:/tmp/Modelfile.tri7b
gathering model components 
Error: 400 Bad Request: invalid model name


In [11]:
# ===== 컨테이너 안에 실제로 들어간 Modelfile 내용 확인 =====
!docker exec ollama cat /tmp/Modelfile.tri7b

FROM /home/kopo17/AI_programming/petmint-rag-chatbot/models/gguf/Tri-7B-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.3


In [12]:
!docker exec ollama ls -la /tmp/Tri-7B-Q4_K_M.gguf

-rw-r--r-- 1 ubuntu ubuntu 4630728384 Sep 29 16:42 /tmp/Tri-7B-Q4_K_M.gguf


In [13]:
# ===== Modelfile 재작성: 컨테이너 안 경로(/tmp/...)로 지정 =====
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write('''FROM /tmp/Tri-7B-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.3
''')

print(open(modelfile_path).read())

FROM /tmp/Tri-7B-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.3



In [14]:
# ===== 다시 복사하고 등록 =====
!docker cp {modelfile_path} ollama:/tmp/Modelfile.tri7b
!docker exec ollama cat /tmp/Modelfile.tri7b
!docker exec ollama ollama create tri-7b -f /tmp/Modelfile.tri7b
!docker exec ollama ollama list | grep tri

Successfully copied 78B (transferred 2.05kB) to ollama:/tmp/Modelfile.tri7b
FROM /tmp/Tri-7B-Q4_K_M.gguf
PARAMETER num_ctx 8192
PARAMETER temperature 0.3
gathering model components ⠋ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠙ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠴ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathe

In [15]:
# ===== Tri-7B 동작 확인 =====
import requests, time

def ask_tri(q):
    t = time.time()
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": "tri-7b",
        "messages": [{"role": "user", "content": q}],
        "stream": False,
    }).json()
    print(f"[{time.time() - t:.1f}초 | 종료사유: {r.get('done_reason')}]")
    print(r["message"]["content"], "\n" + "-" * 60)

ask_tri("안녕하세요")
ask_tri("보험금 청구서류를 접수한 후 얼마나 빨리 받을 수 있나요? 반려동물보험 상담원처럼 답해주세요.")

[3.0초 | 종료사유: stop]
안녕하세요! 만나서 반갑습니다. 오늘 어떤 도움이 필요하신가요? 질문이나 요청이 있으시면 언제든지 말씀해주세요. 저는 당신의 문제를 이해하고 도움을 드리기 위해 여기 있습니다. 즐거운 대화 되시길 바랍니다! 
------------------------------------------------------------
[1.8초 | 종료사유: stop]
안녕하세요! 반려동물보험 관련해서 보험금 청구서류 접수 후 얼마나 빨리 받을 수 있는지에 대해 상담해 드릴게요.

일반적으로 보험금 청구서류 접수 후 보험금을 받기까지 걸리는 시간은 여러 요소에 따라 달라질 수 있어요. 주요 요인은 다음과 같습니다:

1. **서류의 완전성**: 모든 필요한 서류(진료비 영수증, 진단서 등)가 정확하게 제출되었는지 확인합니다. 서류가 미비하면 추가 요청이 있을 수 있어 시간이 더 걸릴 수 있어요.

2. **진료 내용의 검토**: 보험사에서 청구 내용을 검토하고, 보장 조건에 맞는지 확인합니다. 이 과정에서 추가 확인이 필요할 수 있습니다.

3. **보험사의 처리 속도**: 각 보험사의 내부 처리 속도와 정책에 따라 다릅니다. 일부 보험사는 신속하게 처리하는 반면, 다른 곳은 시간이 더 걸릴 수 있어요.

4. **청구 금액**: 소액 청구와 고액 청구에 따라 처리 우선순위가 다를 수 있습니다.

보통, 서류가 완벽하게 제출되고 특별한 문제가 없다면, **1주일에서 2주일 정도** 안에 보험금을 받을 수 있습니다. 하지만 복잡한 경우나 추가 확인이 필요한 경우에는 더 오래 걸릴 수도 있어요.

혹시 더 궁금한 점이 있으시면 언제든지 물어보세요! 예를 들어, 어떤 서류가 필요한지, 청구 과정에서 주의해야 할 점 등이 있다면 자세히 설명해 드릴 수 있어요. 반려동물 건강도 잘 챙기시고, 보험 처리도 원활하게 진행되길 바랍니다! 😊 
------------------------------------------------------------
